# L89 · RAG 完整流水线 — 文档切片 + TF-IDF 检索 + 提示词构建 + 生成

**学习目标**
- 把 L88 的 TF-IDF 检索器组装进完整 RAG 流水线
- 实现 chunk_text() 文档切片
- 实现 format_rag_prompt() 提示词模板
- 跑通：问题 → 检索 → 填充上下文 → 生成回答

## RAG（Retrieval-Augmented Generation）全景

```
用户问题
   ↓
[检索] TF-IDF cosine 搜索知识库  ← aurora.llm.build_tfidf / cosine_retrieve
   ↓
[增强] 把检索到的段落插入提示词
   ↓
[生成] LLM 根据上下文回答
   ↓
回答 + 引用来源
```

**这个流水线的检索部分完全无需 GPU 或预训练模型。**
生成部分可选接 LLM（见 L87），也可以用规则模板演示。

In [ ]:
import numpy as np
from aurora.llm import build_tfidf, cosine_retrieve

In [ ]:
# 知识库（音频 AI 相关，更长的段落）
KNOWLEDGE_BASE = [
    """The Fast Fourier Transform (FFT) is an algorithm that computes the Discrete Fourier Transform
in O(N log N) time. It was developed by Cooley and Tukey in 1965. The FFT works by recursively
splitting the DFT into even and odd sub-problems, known as the Cooley-Tukey radix-2 algorithm.""",

    """The Short-Time Fourier Transform (STFT) applies a windowed FFT to a signal at successive
time steps to produce a spectrogram. It provides a time-frequency representation showing how the
frequency content of a signal changes over time. The window size controls the time-frequency
resolution tradeoff.""",

    """Mel Frequency Cepstral Coefficients (MFCCs) are widely used features in speech recognition.
They are computed by applying a filterbank of triangular filters on the mel scale, taking the
log, and applying a Discrete Cosine Transform. The mel scale approximates the human auditory
system's logarithmic frequency perception.""",

    """CTC (Connectionist Temporal Classification) is a training objective for sequence-to-sequence
models where the alignment between input and output is unknown. It introduces a blank symbol and
sums over all possible alignments using a forward-backward algorithm. CTC is used in speech
recognition models like DeepSpeech.""",

    """Whisper is a speech recognition model from OpenAI trained on 680,000 hours of multilingual
audio. It uses a Transformer encoder-decoder architecture with log-Mel spectrogram inputs.
Whisper can transcribe speech in multiple languages and translate to English.""",

    """The Nyquist-Shannon sampling theorem states that a signal must be sampled at a rate at least
twice its highest frequency component to be reconstructed without aliasing. For example, CD audio
uses 44,100 Hz to capture frequencies up to 22,050 Hz, covering the full range of human hearing.""",
]
print(f'知识库：{len(KNOWLEDGE_BASE)} 个段落')

## ✏️ 任务 1：文档切片

In [ ]:
def chunk_text(text: str, max_words: int = 80, overlap: int = 10) -> list:
    """将长文本切成有重叠的词块。"""
    words = text.split()
    if len(words) <= max_words:
        return [text]
    # ✏️ TODO: 滑动窗口切块
    chunks = []
    step = max_words - overlap
    for i in range(0, len(words), step):
        chunk = ' '.join(words[i:i+max_words])
        chunks.append(chunk)
        if i + max_words >= len(words):
            break
    return chunks

# 测试
long_text = ' '.join(['word'] * 200)
chunks = chunk_text(long_text, max_words=80, overlap=10)
print(f'200 词文本 → {len(chunks)} 个块（max_words=80, overlap=10）')

# 对知识库分块
all_chunks = []
for doc in KNOWLEDGE_BASE:
    all_chunks.extend(chunk_text(doc, max_words=60))
print(f'知识库分块后：{len(all_chunks)} 个 chunk')

## ✏️ 任务 2：RAG 提示词模板

In [ ]:
def format_rag_prompt(query: str, retrieved_docs: list) -> str:
    """将检索到的段落填入提示词模板。"""
    # ✏️ TODO: 构建包含上下文的提示词
    context_parts = []
    for i, (doc, score) in enumerate(retrieved_docs, 1):
        context_parts.append(f'[Source {i} (relevance={score:.2f})]\n{doc}')
    context = '\n\n'.join(context_parts)

    prompt = f"""You are a helpful AI assistant. Answer the question based on the following context.

Context:
{context}

Question: {query}

Answer:""".strip()
    return prompt

In [ ]:
# 构建 TF-IDF 索引
matrix, vocab = build_tfidf(all_chunks)
print(f'索引：{matrix.shape[0]} 个 chunk，词汇量 {matrix.shape[1]}')

# RAG 流水线演示
QUERIES = [
    'How does the FFT algorithm work?',
    'What are MFCCs used for?',
    'How does CTC handle alignment in speech recognition?',
]

for query in QUERIES:
    print(f'\n{"="*55}')
    print(f'问题：{query}')
    print('-'*55)
    results = cosine_retrieve(query, matrix, vocab, all_chunks, top_k=2)
    prompt = format_rag_prompt(query, results)
    # 打印检索结果
    for doc, score in results:
        print(f'  [{score:.3f}] {doc[:80]}...')
    print()
    print('生成提示词（前 200 字符）:')
    print(prompt[:200] + '...')

In [ ]:
# 可选：接入真实 LLM 生成回答
try:
    from transformers import pipeline
    print('（如果已安装 transformers + 模型，可在此生成真实回答）')
    # generator = pipeline('text-generation', model='Qwen/Qwen2.5-0.5B')
    # answer = generator(prompt, max_new_tokens=100)[0]['generated_text']
except ImportError:
    print('transformers 未安装 — RAG 检索部分（无需 GPU）已完整运行 ✅')

## 小结

| 组件 | 技术 | 依赖 |
|---|---|---|
| 文档切片 | 滑动窗口 | 纯 Python |
| 索引构建 | TF-IDF | aurora.llm + NumPy |
| 检索 | 余弦相似度 k-NN | aurora.llm + NumPy |
| 提示词模板 | 字符串格式化 | 纯 Python |
| 生成 | LLM（可选）| transformers（可选）|

**关键洞察**：RAG 的检索质量决定了生成质量。好的检索 > 大的模型。

下一步 L90：Tool-calling Agent——让模型自己决定什么时候调用哪个工具。